In [0]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql import DataFrame
from functools import reduce
from delta.tables import DeltaTable
import requests, zipfile, io
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
from datetime import datetime
import pprint
import os
import json
import time

## CDI

### 1. Verificando os arquivos salvando em uma camada intermediaria

In [0]:
hoje = datetime.today()
menos_10_anos = hoje.replace(year=hoje.year - 10)
menos_10_anos = menos_10_anos.strftime(r"%d/%m/%Y")

#CDI (taxa diária — código 12):
url_bc_selic = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados?formato=json&dataInicial={menos_10_anos}"

response = requests.get(url_bc_selic)

response.raise_for_status()
data_cdi = response.json()

file_path_selic = "/Volumes/workspace/case_spark_cvm/raw/data_cdi_diario/"
df_cdi = spark.createDataFrame(data_cdi)
df_cdi.write.mode('overwrite').parquet(file_path_selic)

### 2. Salvar em camada Bronze Particionada

In [0]:
df_cdi_bronze = spark.read.parquet("/Volumes/workspace/case_spark_cvm/raw/data_cdi_diario/")

df_cdi_bronze = df_cdi_bronze.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)

data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_cdi_bronze.write \
    .mode('overwrite') \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .partitionBy('data_processamento')\
    .format('delta')\
    .save("/Volumes/workspace/case_spark_cvm/bronze/data_cdi_diario/")